# Smart Food Freshness Detection — RGB Camera + Gas Sensor (MQ135 + MQ3) Fusion
### Software Simulation for Project Review

**Reference paper:** Gull et al., *"Smart eNose Food Waste Management System"*, Journal of Sensors, 2021 (MQ4 + MQ135 gas-only system, 92.65% accuracy).

**Our extension:** Add an RGB camera modality and an MQ3 alcohol sensor, then fuse camera + gas predictions to beat gas-only accuracy — proving the value of multimodal sensing before building the ESP32 hardware.

**Important note on data (state this in your review):**
- Image data: **real, public** fresh/rotten fruit photographs (apple, banana, orange), sourced from the Kaggle "Fruits fresh and rotten for classification" dataset (mirrored on GitHub).
- Gas sensor data: **synthetic**, generated using distributions grounded in (a) gas-concentration ranges reported in the reference paper for MQ135, and (b) known food-science behaviour (spoilage → more CO₂ from microbial respiration, more ethanol vapor from fermentation picked up by MQ3, mild NH₃ rise from bacterial protein breakdown). No public dataset pairs MQ135+MQ3 with these exact images, so this synthetic step will be replaced by real logged sensor data once the ESP32 rig is built.


## Step 0 — Install & import libraries

In [ ]:
!pip -q install opencv-python-headless scikit-learn matplotlib pandas numpy joblib
import os, cv2, shutil, random, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
random.seed(42); np.random.seed(42)


## Step 1 — Get the image dataset
We clone the public GitHub mirror of the Kaggle "Fruits fresh and rotten for classification" dataset
(apple / banana / orange, each split into fresh & rotten folders) and build a small, balanced subset
so the notebook runs quickly.


In [ ]:
!git clone --depth 1 https://github.com/Bangkit-JKT2-D/fruits-fresh-rotten-classification.git repo

SRC_ROOT = "repo/Datasets"
DST_ROOT = "subset"
N_TRAIN_PER_CLASS = 150
N_TEST_PER_CLASS = 40
FRESH_DIRS = ["freshapples", "freshbanana", "freshoranges"]
ROTTEN_DIRS = ["rottenapples", "rottenbanana", "rottenoranges"]

def collect_files(split, dirnames):
    files = []
    for d in dirnames:
        folder = os.path.join(SRC_ROOT, split, d)
        for f in os.listdir(folder):
            files.append(os.path.join(folder, f))
    random.shuffle(files)
    return files

def build(split, n_per_class):
    for label, dirnames in [("fresh", FRESH_DIRS), ("rotten", ROTTEN_DIRS)]:
        files = collect_files(split, dirnames)[:n_per_class]
        out_dir = os.path.join(DST_ROOT, split, label)
        os.makedirs(out_dir, exist_ok=True)
        for i, src in enumerate(files):
            ext = os.path.splitext(src)[1]
            dst = os.path.join(out_dir, f"{label}_{i:04d}{ext}")
            shutil.copy(src, dst)
        print(f"{split}/{label}: copied {len(files)} images")

build("train", N_TRAIN_PER_CLASS)
build("test", N_TEST_PER_CLASS)


## Step 2 — RGB camera module simulation
Extract fast, explainable handcrafted features (HSV color histogram, channel stats,
dark/brown-spot ratio, edge density) — the kind of lightweight pipeline an ESP32-CAM +
companion app could realistically compute — then train a classifier.


In [ ]:
IMG_SIZE = (128, 128)

def extract_features(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.resize(img, IMG_SIZE)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hist_h = cv2.calcHist([hsv], [0], None, [16], [0, 180]).flatten(); hist_h /= (hist_h.sum() + 1e-6)
    hist_s = cv2.calcHist([hsv], [1], None, [16], [0, 256]).flatten(); hist_s /= (hist_s.sum() + 1e-6)
    hist_v = cv2.calcHist([hsv], [2], None, [16], [0, 256]).flatten(); hist_v /= (hist_v.sum() + 1e-6)
    means = hsv.reshape(-1, 3).mean(axis=0)
    stds = hsv.reshape(-1, 3).std(axis=0)
    s_chan, v_chan = hsv[:, :, 1], hsv[:, :, 2]
    dark_mask = (v_chan < 90) & (s_chan < 120)
    dark_ratio = dark_mask.sum() / dark_mask.size
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = edges.sum() / (255 * edges.size)
    return np.concatenate([hist_h, hist_s, hist_v, means, stds, [dark_ratio, edge_density]])

def load_split(split):
    X, y, paths = [], [], []
    for label in ["fresh", "rotten"]:
        folder = os.path.join(DST_ROOT, split, label)
        for fname in sorted(os.listdir(folder)):
            fpath = os.path.join(folder, fname)
            feat = extract_features(fpath)
            if feat is not None:
                X.append(feat); y.append(label); paths.append(fpath)
    return np.array(X), np.array(y), paths

X_train_img, y_train, _ = load_split("train")
X_test_img, y_test, test_paths = load_split("test")
print("Train:", X_train_img.shape, " Test:", X_test_img.shape)

img_scaler = StandardScaler()
X_train_img_s = img_scaler.fit_transform(X_train_img)
X_test_img_s = img_scaler.transform(X_test_img)

img_clf = RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42)
img_clf.fit(X_train_img_s, y_train)

img_pred = img_clf.predict(X_test_img_s)
img_proba = img_clf.predict_proba(X_test_img_s)
classes = list(img_clf.classes_)

acc_img = accuracy_score(y_test, img_pred)
print(f"\nIMAGE-ONLY ACCURACY: {acc_img*100:.2f}%")
print(confusion_matrix(y_test, img_pred, labels=classes))
print(classification_report(y_test, img_pred))


## Step 3 — Gas sensor module simulation (MQ135 + MQ3)
Generate one synthetic gas reading **per image**, in the same order, so each sample has a
matched (image, gas) pair — exactly how your real ESP32 rig will capture both modalities
for the same physical food item at the same instant.


In [ ]:
def simulate_gas(labels):
    n = len(labels)
    co2 = np.zeros(n); nh3 = np.zeros(n); mq3 = np.zeros(n)
    for i, lab in enumerate(labels):
        if lab == "fresh":
            co2[i] = np.random.normal(560, 110)
            nh3[i] = np.random.normal(4.3, 0.6)
            mq3[i] = np.random.normal(160, 70)
        else:
            co2[i] = np.random.normal(820, 130)
            nh3[i] = np.random.normal(5.2, 0.7)
            mq3[i] = np.random.normal(370, 110)
    co2 = np.clip(co2, 200, 1300); nh3 = np.clip(nh3, 2.5, 7.5); mq3 = np.clip(mq3, 20, 900)
    return pd.DataFrame({"CO2_ppm": co2, "NH3_ppm": nh3, "MQ3_alcohol_ppm": mq3, "label": labels})

gas_train = simulate_gas(y_train)
gas_test = simulate_gas(y_test)

feat_cols = ["CO2_ppm", "NH3_ppm", "MQ3_alcohol_ppm"]
gas_scaler = StandardScaler()
X_train_gas_s = gas_scaler.fit_transform(gas_train[feat_cols].values)
X_test_gas_s = gas_scaler.transform(gas_test[feat_cols].values)

gas_clf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
gas_clf.fit(X_train_gas_s, gas_train["label"].values)

gas_pred = gas_clf.predict(X_test_gas_s)
gas_proba = gas_clf.predict_proba(X_test_gas_s)

acc_gas = accuracy_score(gas_test["label"].values, gas_pred)
print(f"GAS-ONLY ACCURACY: {acc_gas*100:.2f}%")
print(confusion_matrix(gas_test['label'].values, gas_pred, labels=classes))
print(classification_report(gas_test['label'].values, gas_pred))


## Step 4 — Multimodal fusion (Camera + Gas)
Simple, hardware-realistic **late (decision-level) fusion**: average the two models'
class probabilities per sample, then take the argmax.


In [ ]:
fused_proba = (img_proba + gas_proba) / 2.0
fused_pred = np.array([classes[i] for i in np.argmax(fused_proba, axis=1)])

acc_fused = accuracy_score(y_test, fused_pred)
print(f"FUSED (CAMERA + GAS) ACCURACY: {acc_fused*100:.2f}%")
print(confusion_matrix(y_test, fused_pred, labels=classes))
print(classification_report(y_test, fused_pred))

print("\n--- Summary ---")
print(f"Camera only : {acc_img*100:.2f}%")
print(f"Gas only    : {acc_gas*100:.2f}%")
print(f"Fused       : {acc_fused*100:.2f}%")


## Step 5 — Comparison chart & confusion matrices

In [ ]:
plt.figure(figsize=(6,4.5))
vals = [acc_img*100, acc_gas*100, acc_fused*100]
bars = plt.bar(["Camera only","Gas only","Fused"], vals, color=["#4C72B0","#DD8452","#55A868"])
for b,v in zip(bars, vals):
    plt.text(b.get_x()+b.get_width()/2, v+0.5, f"{v:.1f}%", ha='center', fontweight='bold')
plt.ylabel("Accuracy (%)"); plt.ylim(80,105)
plt.title("Freshness Classification Accuracy: Camera vs Gas vs Fused")
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1,3, figsize=(13,4))
preds = {"Camera only": img_pred, "Gas only": gas_pred, "Fused": fused_pred}
for ax,(name,pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, pred, labels=classes)
    ax.imshow(cm, cmap="Blues"); ax.set_title(name)
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes)
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j,i,cm[i,j],ha="center",va="center",
                     color="white" if cm[i,j]>cm.max()/2 else "black", fontweight="bold")
plt.tight_layout(); plt.show()


## Step 6 — Next steps toward real hardware
1. Replace `simulate_gas()` with real MQ135 + MQ3 ADC readings logged from the ESP32,
   captured at the same instant as each camera frame (same food sample).
2. Re-run this exact pipeline on the real paired dataset — no architecture change needed.
3. Once enough real samples are collected, retrain `img_clf`, `gas_clf`, and re-validate
   the fusion weight (see optional weighted-fusion sweep in the full script).
4. Optionally replace handcrafted image features with a lightweight CNN
   (e.g., MobileNetV2 transfer learning) if the ESP32-CAM sends frames to a phone/edge
   server for classification, since on-device CNN inference on ESP32 alone is limited.
